# 地图可视化：用颜色讲清"哪里多、哪里少"

---

**学习目标**

1. 会用 `Map` 画出中国地图
2. 掌握 `VisualMapOpts` 的**连续**和**分段**两种模式
3. 记住地图可视化最容易踩的坑：**地名必须与地图数据完全一致**
4. 知道版图合规的基本要求

**一句话理解**

地图可视化就是把"数值"翻译成"颜色深浅"。
数据本身还是那几个数字，**地图只是给数字加了一个"位置"维度**，
于是你一眼就能看出"哪一片红、哪一片浅"。


---
### 关于地图合规（重要）

使用地图可视化时，有两条底线必须遵守：

1. **版图必须完整准确**：中国的省级行政区包括 23 省、5 自治区、4 直辖市、
   2 特别行政区，以及**中国台湾**。地图数据里这些都要在，不能漏、不能错画。
2. **使用合规的地图源**：本笔记用的是 pyecharts 自带的 ECharts 中国地图数据
   （内置于库中，不走任何境外瓦片服务），符合国内地图使用要求。
   如果需要做**网页交互地图**（可缩放、可点选），应选用腾讯地图、高德地图、
   百度地图或天地图等合规服务，不要用 Google Maps、OpenStreetMap 等境外源。

> **一个必须注意的细节：合规表述与技术写法不是一回事。**
>
> 中国大陆、中国香港、中国澳门、中国台湾都是中国不可分割的一部分。
> 但在**地图数据里**，必须使用 ECharts 地图内置的**区域标识名**才能匹配上，
> 那里的键名是 `台湾`、`香港`、`澳门`（和 `北京`、`湖北` 一样的写法）。
>
> 所以本笔记的规则是：
>
> - **正文和注释里**：说"中国台湾""中国香港""中国澳门"
> - **`data` 的地名键**：用 `台湾`、`香港`、`澳门`，否则那块区域不会变色
>
> 另外，本笔记里的数字是**合成的练习数据**（见 `data/README.md`），不是真实统计数据，
> 只用于演示画图方法。


In [ ]:
from pyecharts.charts import Map
from pyecharts import options as opts


---
## 一、基本三步走


In [ ]:
# 数据格式：(地名, 数值) 的列表。这是 pyecharts 地图的固定格式
# 注意地名必须用地图内置的标识名，不能加"省/市"后缀
data = [
    ('北京', 99),
    ('上海', 199),
    ('湖南', 299),
    ('广东', 499),
    ('台湾', 399),        # 中国台湾
]

map_chart = Map()
map_chart.add(
    series_name='示例数据',      # 系列名，会显示在图表里
    data_pair=data,             # 数据
    maptype='china',            # 地图类型：china / world / 各省名 / 各国名
)
map_chart.render('地图示例.html')
print('已生成 地图示例.html')


---
## 二、配色：`VisualMapOpts` 是地图的灵魂

没有 `visualmap` 的地图只是一张"哪里有点"的图。
加上它，颜色才会跟数值挂上钩。

### 2.1 连续型（默认）：颜色渐变

适合"数值范围大、想看趋势"的场景。


In [ ]:
map_cont = Map()
map_cont.add('确诊人数', data, 'china')

map_cont.set_global_opts(
    title_opts=opts.TitleOpts(title='连续型视觉映射', pos_left='center'),
    visualmap_opts=opts.VisualMapOpts(
        is_show=True,
        is_piecewise=False,        # False = 连续渐变
        min_=0,                    # 最小值（注意是 min_ 不是 min）
        max_=500,                  # 最大值
        # 颜色从浅到深。颜色数量不固定，pyecharts 会自动插值
        range_color=['#E0F3FF', '#66B2FF', '#0000FF', '#660000'],
    ),
)
map_cont.render('连续型地图.html')
print('已生成 连续型地图.html')


### 2.2 分段型：把数值切成几档

适合"数值跨度极大"的场景。
比如确诊人数从 1 到 60000，如果用连续型，
除了湖北以外的省份会全部挤在同一个浅色区间里，看不出差别。

**切成几档之后，每一档都能有自己的颜色，差异就出来了。**


In [ ]:
map_piece = Map()
map_piece.add('示例数据', data, 'china')

map_piece.set_global_opts(
    title_opts=opts.TitleOpts(title='分段型视觉映射', pos_left='center'),
    visualmap_opts=opts.VisualMapOpts(
        is_show=True,
        is_piecewise=True,         # True = 分段
        pieces=[
            # 每一段的写法：{"min": 最小值, "max": 最大值, "label": 显示的说明, "color": 颜色}
            {'min': 1, 'max': 9, 'label': '1-9 人', 'color': '#CCFFFF'},
            {'min': 10, 'max': 99, 'label': '10-99 人', 'color': '#FFCC99'},
            {'min': 100, 'max': 299, 'label': '100-299 人', 'color': '#FF6666'},
            {'min': 300, 'max': 500, 'label': '300-500 人', 'color': '#990033'},
        ],
    ),
)
map_piece.render('分段型地图.html')
print('已生成 分段型地图.html')


> **注意**：`pieces` 里最后一段如果只想写下限（表示"某某以上"），
> 就省略 `max` 键。比如 `{"min": 2500, "label": "2500 人以上", "color": "#FF00FF"}`。
>
> **常见错误**：颜色值多写了一个 `#`。比如 `"#0000FF"` 写成 `"##0000FF"`，
> 图表会渲染成黑色或直接失效。原始笔记里就有这个笔误，已经修正。


---
## 三、最容易踩的坑：地名必须"对得上"

pyecharts 是靠**字符串匹配**把数据和地图上的区域关联起来的。
名字对不上，这个区域就**不会有颜色**，而且**不会报错** —— 你只会看到一块灰。

### 正确的地名写法


In [ ]:
# 正确的名称列表（pyecharts 中国地图使用的标准区域标识名）
china_regions = [
    # 23 省
    '河北', '山西', '辽宁', '吉林', '黑龙江', '江苏', '浙江', '安徽', '福建',
    '江西', '山东', '河南', '湖北', '湖南', '广东', '海南', '四川', '贵州',
    '云南', '陕西', '甘肃', '青海', '台湾',          # 中国台湾
    # 5 自治区
    '内蒙古', '广西', '西藏', '宁夏', '新疆',
    # 4 直辖市
    '北京', '天津', '上海', '重庆',
    # 2 特别行政区
    '香港', '澳门',                                  # 中国香港、中国澳门
]

print(f'中国共有 {len(china_regions)} 个省级行政区')
print('（23 省 + 5 自治区 + 4 直辖市 + 2 特别行政区 = 34）')


### 常见的名称错误对照表

| 错误写法 | 正确写法 | 说明 |
|---|---|---|
| `湖北省` | `湖北` | **不要加"省"字** |
| `北京市` | `北京` | 不要加"市"字 |
| `内蒙` | `内蒙古` | 简称不被识别 |
| `新疆维吾尔自治区` | `新疆` | 不要写全称 |
| `广西壮族自治区` | `广西` | 不要写全称 |
| `台湾省` | `台湾` | 地图标识名不带后缀（中国台湾） |
| `香港特别行政区` | `香港` | 地图标识名不带后缀（中国香港） |

**原始笔记里有个典型的坑**：

```python
from operator import concat
province_name = concat(province_data["name"], "省")    # 给名字硬加上"省"
```

`operator.concat` 确实是"拼接"函数，但**给每个省名都加"省"字是错的**：

- 直辖市加"省" → `北京市省`，匹配不上
- 自治区加"省" → `内蒙古省`，匹配不上
- 特别行政区加"省" → 也不对

**正确做法：直接用数据里原来的名字，不要做任何拼接。**


In [ ]:
# 演示：匹配不上的地名会发生什么（不会报错，只是那块地没颜色）
bad_data = [
    ('北京市', 100),      # 错了
    ('湖北省', 200),      # 错了
    ('北京', 99),         # 对了
    ('湖北', 199),        # 对了
]

map_match = Map()
map_match.add('名称匹配演示', bad_data, 'china')
map_match.set_global_opts(
    title_opts=opts.TitleOpts(
        title='名称匹配演示：只有"北京""湖北"会变色',
        subtitle='注意看：写成"北京市""湖北省"的那两条数据被忽略了，而且没有任何报错',
        pos_left='center',
    ),
    visualmap_opts=opts.VisualMapOpts(min_=0, max_=200),
)
map_match.render('名称匹配演示.html')
print('已生成 名称匹配演示.html')
# 教训：地图出来"缺一块"的时候，第一件事就是检查地名拼写。


---
## 四、其他常用配置


In [ ]:
map_full = Map()
map_full.add(
    series_name='示例数据',
    data_pair=data,
    maptype='china',
    is_map_symbol_show=False,        # 不显示红色小标记点
    label_opts=opts.LabelOpts(is_show=True, font_size=8),   # 是否在地图上直接显示地名
    itemstyle_opts=opts.ItemStyleOpts(
        border_color='#666',         # 区域边框颜色
        border_width=0.5,
    ),
)

map_full.set_global_opts(
    title_opts=opts.TitleOpts(
        title='完整配置示例',
        subtitle='数据为练习用合成数据',
        pos_left='center',
        pos_top='2%',
    ),
    legend_opts=opts.LegendOpts(is_show=False),
    toolbox_opts=opts.ToolboxOpts(is_show=True),
    visualmap_opts=opts.VisualMapOpts(
        is_show=True,
        is_piecewise=True,
        pos_left='10%',
        pos_bottom='10%',
        pieces=[
            {'min': 1, 'max': 99, 'label': '少于 100', 'color': '#FFE4B5'},
            {'min': 100, 'max': 299, 'label': '100 - 299', 'color': '#FFA07A'},
            {'min': 300, 'max': 999, 'label': '300 以上', 'color': '#CD5C5C'},
        ],
    ),
    tooltip_opts=opts.TooltipOpts(formatter='{b}: {c}'),      # {b}=地名 {c}=数值
)
map_full.render('完整配置地图.html')
print('已生成 完整配置地图.html')


### 其他地图类型

`maptype` 参数除了 `'china'`，还支持：

| maptype 值 | 效果 |
|---|---|
| `'china'` | 中国全国 |
| `'world'` | 世界地图 |
| `'湖北'` / `'广东'` 等 | **单个省份**，显示到市级 |
| `'北京'` 等直辖市 | 显示到区级 |
| `'China'` + 城市名 | 需要额外的地图数据文件 |


In [ ]:
# 省级地图：显示到市
hubei_data = [
    ('武汉', 500),
    ('黄石', 200),
    ('十堰', 150),
    ('宜昌', 300),
    ('襄阳', 250),
]

map_hubei = Map()
map_hubei.add('湖北省各市数据', hubei_data, '湖北')
map_hubei.set_global_opts(
    title_opts=opts.TitleOpts(title='湖北省市级地图', pos_left='center'),
    visualmap_opts=opts.VisualMapOpts(min_=0, max_=500),
)
map_hubei.render('湖北省地图.html')
print('已生成 湖北省地图.html')


---
## 五、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 地名加了"省/市" | 区域没颜色，**且不报错** | 用标准简称：`湖北` 而非 `湖北省` |
| 颜色写成 `##0000FF` | 渲染异常 / 变黑 | 一个 `#` 就够 |
| 忘了 `visualmap_opts` | 地图只有标记点，没有颜色差异 | 加上视觉映射 |
| `is_piecewise=True` 但没给 `pieces` | 只有默认的一档 | 显式提供 `pieces` 列表 |
| 数值跨度大还用连续型 | 除个别省外全是浅色，看不出差别 | 改用分段型 |
| 用 `min` / `max` 而不是 `min_` / `max_` | 参数不生效 | 注意下划线后缀 |
| 数据里漏了行政区 | 版图不完整 | 覆盖全部 34 个省级行政区 |
| 用境外地图服务做网页地图 | 合规问题 | 用腾讯/高德/百度/天地图 |
| 数据传成了字典 | 报错 | 必须是 `(地名, 数值)` 的**列表** |

---

## 六、动手练习

**练习 1**：把上面的 `data` 换成"你熟悉的 8 个省份 + 随机数值"，画一张分段地图。

**练习 2**：故意写错 3 个地名，观察地图有什么变化，验证"不报错但缺一块"这个现象。

**练习 3**：画一张你自己家乡的**市级地图**（`maptype='省份名'`），
数据填 8 个左右的市。

**练习 4**：用 `data/地图数据/疫情统计.csv` 里的真实列（虽然也是合成数据），
画一张"治愈人数"的分段地图。提示：
```python
import csv
with open(path, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
data = [(r['省份'], int(r['治愈人数'])) for r in rows]
```

---

**答案思路（先自己做）**

```python
# 练习 4
import csv
from pathlib import Path

csv_file = Path('data/地图数据/疫情统计.csv')
with open(csv_file, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

cured_data = [(r['省份'], int(r['治愈人数'])) for r in rows[:10]]

m = Map()
m.add('治愈人数', cured_data, 'china')
m.set_global_opts(
    title_opts=opts.TitleOpts(title='治愈人数分布', pos_left='center'),
    visualmap_opts=opts.VisualMapOpts(is_piecewise=True, pieces=[
        {'min': 0, 'max': 99, 'label': '0-99', 'color': '#E0F3FF'},
        {'min': 100, 'max': 499, 'label': '100-499', 'color': '#66B2FF'},
        {'min': 500, 'label': '500 以上', 'color': '#0033CC'},
    ]),
)
m.render('治愈人数地图.html')
```
